# JSON Data Modelling

## Overview
Choosing between JSON columns and normalised relational tables is one of the most consequential schema decisions in a modern healthcare or financial database. This notebook provides a decision framework and demonstrates both patterns.

**The fundamental rule:**
> Use relational columns for data you **query**, **aggregate**, or **join on**.
> Use JSON for data you **store and retrieve as a blob**, or that has a **genuinely variable schema**.

**Anti-patterns to avoid:**

| Anti-pattern | Problem | Fix |
|---|---|---|
| All data as JSON | Loses type safety, indexes, FKs | Normalise structured, queryable fields |
| JSON for frequently-updated fields | Every update rewrites the whole blob | Use dedicated typed columns |
| Arrays instead of junction tables | No FK, no metadata, no deduplication | Use a junction table |
| Deeply nested JSON (>3 levels) | Hard to query and update | Flatten or normalise |
| No index on queried JSON field | Full table scan on every filter | Add expression index |

---

In [ ]:
print("=== JSON vs relational columns: when to use each ===")
print()

decision_table = [
    ("Use relational columns when:",   [
        "You query/filter on the value frequently (WHERE age > 50)",
        "You need referential integrity (FOREIGN KEY)",
        "You aggregate the value (SUM, AVG, COUNT, GROUP BY)",
        "The schema is stable and well-known",
        "You need per-column indexes for performance",
        "You join on the value (JOIN patients ON province = ...)",
    ]),
    ("Use JSON when:",   [
        "Schema varies per row (patient A has 3 fields, patient B has 10)",
        "The structure changes frequently (new fields added often)",
        "The data is consumed as a blob (sent to frontend as-is)",
        "You store optional, sparse metadata (preferences, tags)",
        "The JSON is an external API payload stored verbatim",
        "You need array/nested structure that would require junction tables",
    ]),
    ("Avoid JSON when:",  [
        "You need to filter or sort by a JSON field in most queries",
        "The value needs a type constraint (NUMERIC, NOT NULL, CHECK)",
        "The field participates in JOINs or foreign keys",
        "Multiple services need to update individual JSON fields atomically",
        "The data belongs in a dedicated NoSQL store (use the right tool)",
    ]),
]
for section, points in decision_table:
    print(f"  {section}")
    for p in points:
        print(f"    - {p}")
    print()


---
## Anti-patterns in depth

In [ ]:
print("=== JSON anti-patterns to avoid ===")
print()

antipatterns = [
    ("The EAV in JSON anti-pattern",
     "Storing all patient attributes as {key: 'age', value: '45'} JSON objects",
     "You lose type safety and indexability; use actual columns for known attributes"),
    ("JSON for data that changes frequently",
     "Using JSON for a field like 'last_login' that updates on every request",
     "Every update rewrites the entire JSON blob; use a dedicated timestamp column"),
    ("Deeply nested JSON (>3 levels)",
     "clinical.history.visits.2023.appointments[0].doctor.name",
     "Hard to query, hard to index, hard to update; flatten or normalise"),
    ("JSON arrays as sets (no deduplication)",
     "conditions: ['Hypertension', 'Hypertension', 'HTN']",
     "JSON has no UNIQUE constraint; use a normalised table with a unique index"),
    ("Missing index on heavily-queried JSON field",
     "WHERE clinical->>'smoker' = 'true' on 5M rows with no index",
     "Create expression index: CREATE INDEX ON patients ((clinical->>'smoker'))"),
    ("Mixing JSON and relational for the same data",
     "province in both patients.province column AND demographics->>'province'",
     "Single source of truth; pick one; dual storage leads to sync bugs"),
    ("Storing JSON strings-of-strings",
     "conditions column contains a JSON string that is itself a JSON-encoded string",
     "Always store valid JSON directly; never double-encode"),
]
print(f"  {'Anti-pattern':<38s}  {'Example':<46s}  Fix")
print("  " + "-"*110)
for name, example, fix in antipatterns:
    print(f"  {name:<38s}  {example:<46s}  {fix}")

print()
# Demonstrate refactoring
print("\n=== Refactoring example: JSON conditions array → relational ===")
print("""
  -- BEFORE (JSON):
  SELECT p.full_name, json_extract(clinical, '$.conditions') AS conditions
  FROM   patients;
  -- Querying is awkward; no index; no FK; duplicates possible

  -- AFTER (normalised):
  CREATE TABLE patient_conditions (
      patient_id   TEXT NOT NULL REFERENCES patients(patient_id),
      condition    TEXT NOT NULL,
      icd10        TEXT,
      diagnosed_at DATE,
      PRIMARY KEY (patient_id, condition)
  );

  -- Clean queries:
  SELECT p.full_name, c.condition
  FROM   patients p
  JOIN   patient_conditions c ON c.patient_id = p.patient_id
  WHERE  c.condition = 'Hypertension';

  -- But KEEP JSON for truly variable/sparse data:
  --   preferences, notification_settings, audit_metadata, raw_api_payloads
""")


---
## Hybrid schema: the right tool for each data type

In [ ]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

# Healthcare dataset: patients with JSON clinical data
conn.executescript("""
CREATE TABLE patients (
    patient_id   TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    demographics JSON NOT NULL,   -- age, dob, province, contact info
    clinical     JSON NOT NULL,   -- conditions, allergies, vitals history
    preferences  JSON             -- notification prefs, language, etc.
);

CREATE TABLE lab_results (
    lab_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    result_date TEXT NOT NULL,
    tests       JSON NOT NULL    -- array of {test, value, unit, ref_range, flag}
);

CREATE TABLE medications (
    med_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    prescribed  TEXT NOT NULL,
    details     JSON NOT NULL    -- drug, dose, frequency, prescriber, side_effects
);
""")

patients = [
  ('P001','Alice Ngata',
   json.dumps({'age':45,'dob':'1980-03-15','province':'NB','contact':{'phone':'506-555-0101','email':'alice@email.com'},'insurance':{'plan':'premium','id':'INS-00123'}}),
   json.dumps({'conditions':['Hypertension','Hyperlipidaemia'],'allergies':[],'vitals':[{'date':'2023-04-10','bp':'142/88','weight_kg':72},{'date':'2023-10-01','bp':'128/82','weight_kg':70}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P002','Bob Chen',
   json.dumps({'age':52,'dob':'1972-07-22','province':'ON','contact':{'phone':'416-555-2233','email':'bob@email.com'},'insurance':{'plan':'standard','id':'INS-00456'}}),
   json.dumps({'conditions':['Hypertension','Type 2 Diabetes'],'allergies':['Penicillin'],'vitals':[{'date':'2023-05-15','bp':'148/92','weight_kg':88},{'date':'2024-01-10','bp':'138/86','weight_kg':86}],'smoker':True}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':True},'portal_enabled':True})),
  ('P003','Fatima Rashid',
   json.dumps({'age':38,'dob':'1986-11-05','province':'BC','contact':{'phone':'604-555-9900','email':'fatima@email.com'},'insurance':{'plan':'premium','id':'INS-00789'}}),
   json.dumps({'conditions':['Asthma','Obesity'],'allergies':['Sulfa drugs','NSAIDs'],'vitals':[{'date':'2023-08-20','bp':'148/92','weight_kg':95}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':False,'sms':True},'portal_enabled':False})),
  ('P004','James MacLeod',
   json.dumps({'age':61,'dob':'1963-01-30','province':'NS','contact':{'phone':'902-555-7788','email':'james@email.com'},'insurance':{'plan':'standard','id':'INS-01011'}}),
   json.dumps({'conditions':['Type 2 Diabetes'],'allergies':[],'vitals':[{'date':'2023-09-01','bp':'118/76','weight_kg':80}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P005','Mei-Ling Tran',
   json.dumps({'age':58,'dob':'1966-08-19','province':'QC','contact':{'phone':'514-555-1122','email':'mei@email.com'},'insurance':{'plan':'premium','id':'INS-01213'}}),
   json.dumps({'conditions':['Type 2 Diabetes','Hypertension','CKD Stage 3'],'allergies':['Metformin'],'vitals':[{'date':'2023-11-10','bp':'155/96','weight_kg':65},{'date':'2024-02-01','bp':'145/90','weight_kg':64}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':True,'sms':True},'portal_enabled':True})),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", patients)

lab_rows = [
  ('P001','2023-10-01', json.dumps([
    {'test':'HbA1c','value':7.2,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':68,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'},
    {'test':'LDL','value':3.1,'unit':'mmol/L','ref_range':'<2.6','flag':'H'}])),
  ('P002','2024-01-10', json.dumps([
    {'test':'HbA1c','value':8.4,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':55,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Creatinine','value':115,'unit':'umol/L','ref_range':'62-106','flag':'H'}])),
  ('P004','2023-09-01', json.dumps([
    {'test':'HbA1c','value':7.8,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':72,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'}])),
  ('P005','2024-02-01', json.dumps([
    {'test':'HbA1c','value':9.1,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':38,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Potassium','value':5.8,'unit':'mmol/L','ref_range':'3.5-5.0','flag':'H'}])),
]
conn.executemany("INSERT INTO lab_results (patient_id,result_date,tests) VALUES (?,?,?)", lab_rows)

med_rows = [
  ('P001','2023-01-15', json.dumps({'drug':'Lisinopril','dose':'10mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['dizziness','dry cough']})),
  ('P001','2023-01-15', json.dumps({'drug':'Atorvastatin','dose':'40mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['myalgia']})),
  ('P002','2022-06-01', json.dumps({'drug':'Metformin','dose':'500mg','frequency':'BID','prescriber':'Dr. Pham','side_effects':['nausea','diarrhoea']})),
  ('P002','2022-06-01', json.dumps({'drug':'Lisinopril','dose':'5mg','frequency':'once daily','prescriber':'Dr. Pham','side_effects':[]})),
  ('P003','2021-09-10', json.dumps({'drug':'Salbutamol','dose':'2.5mg','frequency':'PRN','prescriber':'Dr. Singh','side_effects':['tremor','palpitations']})),
  ('P005','2023-03-01', json.dumps({'drug':'Insulin glargine','dose':'20 units','frequency':'nocte','prescriber':'Dr. Pham','side_effects':['hypoglycaemia']})),
]
conn.executemany("INSERT INTO medications (patient_id,prescribed,details) VALUES (?,?,?)", med_rows)
conn.commit()

print("Healthcare JSON dataset ready:")
for tbl in ("patients","lab_results","medications"):
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {n} rows")

# Preview one JSON column
row = conn.execute("SELECT full_name, demographics FROM patients LIMIT 1").fetchone()
print(f"\nSample demographics JSON for {row['full_name']}:")
print(json.dumps(json.loads(row['demographics']), indent=2))


print("=== Hybrid schema: relational + JSON for the right use cases ===")
print()

print("Our patients table uses a hybrid approach:")
schema = [
    ("patient_id",  "TEXT PRIMARY KEY",    "relational", "Stable identifier — always queried, always indexed"),
    ("full_name",   "TEXT NOT NULL",        "relational", "Filtered, sorted, displayed — needs index"),
    ("demographics","JSON",                 "json",       "Sparse contact/insurance data — consumed as blob"),
    ("clinical",    "JSON",                 "json",       "Variable conditions/allergies/vitals — schema varies"),
    ("preferences", "JSON",                 "json",       "Optional notification settings — rarely queried"),
]
print(f"  {'Column':<14s}  {'Type':<18s}  {'Storage':<12s}  Rationale")
print("  " + "-"*80)
for col, typ, storage, reason in schema:
    print(f"  {col:<14s}  {typ:<18s}  {storage:<12s}  {reason}")

print()
print("Recommended pattern for healthcare records:")
recommended = [
    ("Patient identity",      "Relational columns",         "patient_id, name, dob, province — always queried"),
    ("Structured vitals",     "Separate vitals table",      "One row per visit — aggregatable, indexable"),
    ("Conditions/allergies",  "Normalised junction table",  "patient_conditions — FK-enforced, deduplicated"),
    ("Medications",           "Normalised medications table","FK to drug catalogue, dose/frequency as columns"),
    ("Raw lab results",       "JSON or separate table",     "JSON if format varies; relational if fixed schema"),
    ("Preferences/settings",  "JSON",                       "Sparse, optional, consumed as blob"),
    ("Audit metadata",        "JSON or JSONB",              "Flexible key-value log; rarely queried by field"),
    ("External API payloads", "JSON/JSONB verbatim",        "Store raw; extract columns as needed later"),
]
print(f"  {'Data type':<26s}  {'Recommendation':<28s}  Why")
print("  " + "-"*80)
for data, rec, why in recommended:
    print(f"  {data:<26s}  {rec:<28s}  {why}")


---
## Common Pitfalls

**1. Defaulting to JSON for everything (document-database thinking)**
Developers familiar with MongoDB may store entire records as JSON. In PostgreSQL, this loses all the benefits of relational modelling: type safety, FK constraints, clean aggregations, and efficient indexes. Use JSON for genuinely variable or sparse data — not as a substitute for proper relational design.

**2. Storing frequently-updated fields in JSON**
Every UPDATE to a JSONB column rewrites the entire JSONB value (PostgreSQL does MVCC — a new row version is created). Updating `last_login` inside a JSONB preferences object on every login writes a new version of the entire column. Use a dedicated `last_login TIMESTAMPTZ` column for frequently-updated scalars.

**3. Using JSON arrays instead of a junction table for many-to-many**
Storing patient conditions as `['Hypertension', 'Diabetes']` in a JSON array makes it impossible to add metadata (diagnosed_at, severity, icd10_code) without restructuring the entire array. A `patient_conditions` junction table adds this cleanly and allows indexing, FK enforcement, and clean aggregation.

**4. No CHECK constraint to validate JSON structure**
Without a constraint, any valid JSON (including `{}` or `null`) can be stored. Add a CHECK: `CHECK (jsonb_typeof(clinical->'conditions') = 'array')` to enforce structural assumptions your queries depend on.

**5. Evolving JSON schema without migration**
Adding a new key to JSON is easy — but querying it assumes it exists in all rows. Old rows without the new key return NULL. Use `COALESCE(col->>'new_key', 'default')` in queries until a migration backfills the value for all rows.

**6. Conflating JSON storage with JSON query performance**
Storing data as JSON is fast. Querying it on a non-indexed JSON field at scale is slow. Always consider indexing when the JSON field appears in WHERE, ORDER BY, or JOIN.


---
*sql_methods_library - Samantha McGarrigle*